In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict


plt.style.use('seaborn')

In [2]:
input_folder = "../build/CPlexCutsPower_20.05.2023_Сб_22-28"

In [3]:
def search_patterns(filename_pattern, name_pattern, pattern_list, source_dict):
    """
    Iterates all filenames which names satisfy filename_pattern. Retrieves problem name from 
    filename and search for every pattern in pattern_list. Then adds found values to source_dict
    Arguments:
        filename_pattern: regex pattern to select filenames
        name_pattern: regex pattern to retrieve problem name
        pattern_list: list consisting of tuples (pattern, group_name, column_name).
            all patterns contains named group, group_name used to retrieve result from groupdict object
            column_name used to add result to source_dict
        source_dict: dict to add found values
    """
    for filename in glob.glob(filename_pattern):
        match = re.search(name_pattern, filename)
        problem_name = match[0]
        with open(filename, "r") as file:
            file_content = file.read()
            if "terminate..." not in file_content:
                for pattern, group_name, column_name in pattern_list:
                    match = re.search(pattern, file_content)
                    if match:
                        found = match.groupdict()[group_name]
                        source_dict[problem_name][column_name] = found


def parse_logs(logs_dir: str):
    """ 
    Parse logs from logs_dir, return df with parse results. For results with cuts it retrieves
    total time, time for generating cuts, duality gap decrease, ILP solution value. For baseline
    results it parse total time and ILP solution value
    """
    source_dict = defaultdict(dict)

    # Parse results with cuts
    # note: regex used for number with exponent: [-+]*\d*\.*\d+([eE][-+]?\d+)?
    pattern_total_time = r"Total \(root\+branch&cut\) =\s*(?P<total_time>\d+\.\d+)"
    pattern_duality_gap = r"Duality gap decrease \(ILP\): (?P<gap>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    pattern_time_cuts = r"total time: (?P<time_cuts>\d+\.\d+)"
    pattern_ilp_value_cuts = r"\nILP solution value:\s*(?P<ilp_solution_value>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"

    col_cplex_time_after_cuts = "Time cplex after adding cuts"
    col_duality_gap_decrease = "Duality gap decrease (ILP)"
    col_time_generating_cuts = "Time generating cuts"
    col_ilp_value_with_cuts = "ILP solution value with cuts"

    pattern_list_cuts = (
        (pattern_total_time, "total_time", col_cplex_time_after_cuts),
        (pattern_duality_gap, "gap", col_duality_gap_decrease),
        (pattern_time_cuts, "time_cuts", col_time_generating_cuts),
        (pattern_ilp_value_cuts, "ilp_solution_value", col_ilp_value_with_cuts)
    )
    search_patterns(
        filename_pattern = f"{logs_dir}/*gz_log.txt",
        name_pattern = r"(?<=/)[^/]+(?=.mps.gz_log.txt)",
        pattern_list = pattern_list_cuts,
        source_dict = source_dict
    )

    # Parse baseline results (cplex)
    pattern_time_cplex_baseline = r"Total \(root\+branch&cut\) =\s*(?P<total_baseline_time>\d+\.\d+)"
    pattern_ilp_value_baseline = r"\nILP solution value:\s*(?P<ilp_solution_value>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    
    col_cplex_time_baseline = "Time cplex baseline"
    col_ilp_value_baseline = "ILP solution value baseline"
    
    pattern_list_baseline = (
        (pattern_time_cplex_baseline, "total_baseline_time", col_cplex_time_baseline),
        (pattern_ilp_value_baseline, "ilp_solution_value", col_ilp_value_baseline)
    )

    
    search_patterns(
        filename_pattern = f"{logs_dir}/*gz_b_log.txt",
        name_pattern = r"(?<=/)[^/]+(?=.mps.gz_b_log.txt)",
        pattern_list = pattern_list_baseline,
        source_dict = source_dict
    )

    # Construct dataframe
    result_df = pd.DataFrame(source_dict).transpose()

    result_df[col_cplex_time_after_cuts] = result_df[col_cplex_time_after_cuts].astype(float)
    result_df[col_duality_gap_decrease] = result_df[col_duality_gap_decrease].astype(float)
    result_df[col_time_generating_cuts] = result_df[col_time_generating_cuts].astype(float)
    result_df[col_ilp_value_with_cuts] = result_df[col_ilp_value_with_cuts].fillna(0).astype(float).astype(int).replace(0, np.nan)

    result_df[col_cplex_time_baseline] = result_df[col_cplex_time_baseline].astype(float)
    result_df[col_ilp_value_baseline] = result_df[col_ilp_value_baseline].fillna(0).astype(float).astype(int).replace(0, np.nan)

    # Add total time with cuts
    col_time_cuts_plus_cplex = "Time cuts + cplex"
    result_df[col_time_cuts_plus_cplex] = result_df[col_cplex_time_after_cuts] + result_df[col_time_generating_cuts]

    # Order columns
    columns_order = [col_time_generating_cuts, col_cplex_time_after_cuts, col_time_cuts_plus_cplex,
                     col_duality_gap_decrease, col_ilp_value_with_cuts,
                     col_cplex_time_baseline, col_ilp_value_baseline]
    result_df = result_df[columns_order]
    return result_df

result_df = parse_logs(input_folder)
result_df

,Time generating cuts,Time cplex after adding cuts,Time cuts + cplex,Duality gap decrease (ILP),ILP solution value with cuts,Time cplex baseline,ILP solution value baseline
reblock115,67.364600,8182.44,8249.804600,2.573150e-03,-36800600.0,4105.16,-36800600.0
glass-sc,0.683623,3119.84,3120.523623,2.387690e-03,23.0,2727.64,23.0
neos859080,0.407212,36.86,37.267212,NaN,NaN,56.74,NaN
pb-market-split8-70-4,0.087287,NaN,NaN,NaN,NaN,NaN,NaN
cod105_vc1024,178.600000,25.18,203.780000,-7.347660e-15,-12.0,26.92,-12.0
seymour,60.886000,12559.28,12620.166000,2.109330e-02,423.0,NaN,NaN
enlight_hard,0.547682,0.01,0.557682,4.864860e-01,37.0,0.02,37.0


In [14]:
def show_transposed(result_df):
    return result_df.astype(str).replace('nan',np.nan).transpose()

show_transposed(result_df)

,reblock115,glass-sc,neos859080,pb-market-split8-70-4,cod105_vc1024,seymour,enlight_hard
Time generating cuts,67.3646,0.683623,0.407212,0.0872868,178.6,60.886,0.547682
Time cplex after adding cuts,8182.44,3119.84,36.86,NaN,25.18,12559.28,0.01
Time cuts + cplex,8249.8046,3120.523623,37.267212,NaN,203.78,12620.166000000001,0.557682
Duality gap decrease (ILP),0.00257315,0.00238769,NaN,NaN,-7.34766e-15,0.0210933,0.486486
ILP solution value with cuts,-36800600.0,23.0,NaN,NaN,-12.0,423.0,37.0
Time cplex baseline,4105.16,2727.64,56.74,NaN,26.92,NaN,0.02
ILP solution value baseline,-36800600.0,23.0,NaN,NaN,-12.0,NaN,37.0
